<a href="https://colab.research.google.com/github/cathrineq/python-ai-Tarasova-Kate/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- [`currency_rates.csv`](https://github.com/cathrineq/python-ai-Tarasova-Kate/blob/main/data/currency_rates.csv) — курсы обмена валют по времени (Wikidata P2284, P585, P580)
- [`countries_currencies.csv`](https://github.com/cathrineq/python-ai-Tarasova-Kate/blob/main/data/countries_currencies.csv) — страны и их официальные валюты (Wikidata P38, P1082, P2046)

**Что мы делаем:**
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файлы в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [2]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

repo = "python-ai-Tarasova-Kate"  # ← изменено: имя вашего репозитория
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):          # всегда проверяет /content/python-ai-Tarasova-Kate
    !git clone -q https://github.com/cathrineq/python-ai-Tarasova-Kate.git  # ← изменено: ваш URL репозитория

if os.getcwd() != repo_path:               # точное сравнение, не endswith
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

/content/python-ai-Tarasova-Kate
✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-Tarasova-Kate


## 📥 [2A] Простое чтение CSV-файлов в pandas

Сначала просто прочитаем оба CSV-файла в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [3]:
# 🐱 Шаг 2A. Чтение CSV-файлов в pandas

import pandas as pd
import os

print("🔍 Проверка доступных файлов в data/:")
if os.path.exists("data"):
    print(os.listdir("data"))
else:
    print("⚠️ Папка data/ не найдена!")

# Инициализируем переменные
df_rates = None
df_countries = None
df_currency = None

# Пробуем загрузить новые файлы
if os.path.exists("data/currency_rates.csv"):
    df_rates = pd.read_csv("data/currency_rates.csv")
    print("✅ Загружено df_rates:", len(df_rates), "строк")

if os.path.exists("data/countries_currencies.csv"):
    df_countries = pd.read_csv("data/countries_currencies.csv")
    print("✅ Загружено df_countries:", len(df_countries), "строк")

# Если новые файлы не найдены — используем исходный
if df_rates is None and df_countries is None:
    df_currency = pd.read_csv("data/currency.csv")
    print("✅ Загружено df_currency:", len(df_currency), "строк")

🔍 Проверка доступных файлов в data/:
['.gitkeep', 'countries_currencies.rq', 'currency_rates.csv', 'examples', 'README.md', 'currency_rates.rq', 'countries_currencies.csv']
✅ Загружено df_rates: 2346 строк
✅ Загружено df_countries: 803 строк


## 🧹 [2B] Очистка и переименование столбцов

В CSV-файлах из Wikidata есть **технические столбцы**, которые нужно обработать:

### Файл `currency_rates.csv`:
- Столбец `currency` с URL — **удаляем** (оставляем только `currencyLabel`)
- `currencyLabel → currency`, `unitLabel → unit` (убираем постфикс `Label`)
- Числовые: `price`, `year`, `startYear` → приводим к `float`/`int`

### Файл `countries_currencies.csv`:
- Столбцы `country`, `currency` с URL — **удаляем** (оставляем `*Label`)
- `countryLabel → country`, `currencyLabel → currency`
- Числовые: `population`, `area` → приводим к `int`/`float`

> ⚠️ **Важно:** URL Wikidata можно сохранить для отладки, но для анализа удобнее работать с читаемыми названиями.

In [4]:
# 🧹 Шаг 2B. Очистка и переименование столбцов

import pandas as pd

# ========== ФАЙЛ A: Курсы валют (df_rates) ==========
if df_rates is not None:
    print("\n🧹 Очистка df_rates (курсы валют)...")

    # 🔗 НЕ удаляем URL — переименовываем для удобства
    df_rates = df_rates.rename(columns={"currency": "URL"}, errors="ignore")

    # Переименовываем Label-столбцы
    df_rates = df_rates.rename(columns={
        "currencyLabel": "currency",
        "unitLabel": "unit"
    }, errors="ignore")

    # 📊 ИЗМЕРЯЕМ заполненность необязательных полей (ПЕРЕД fillna!)
    print("\n📈 Заполненность полей в df_rates:")
    if "year" in df_rates.columns:
        print(f"  • year: {df_rates['year'].notna().mean() * 100:.1f}% строк с годом")
    if "startYear" in df_rates.columns:
        print(f"  • startYear: {df_rates['startYear'].notna().mean() * 100:.1f}% строк с годом начала")
    if "shortName" in df_rates.columns:
        print(f"  • shortName: {df_rates['shortName'].notna().mean() * 100:.1f}% валют с коротким кодом")

    # Числовые столбцы
    df_rates["price"] = pd.to_numeric(df_rates["price"], errors="coerce")  # float, NaN остаётся

    # ⚠️ year/startYear: НЕ заменяем NaN на 0 (год 0 — бессмысленно)
    if "year" in df_rates.columns:
        df_rates["year"] = pd.to_numeric(df_rates["year"], errors="coerce")  # остаётся float с NaN
    if "startYear" in df_rates.columns:
        df_rates["startYear"] = pd.to_numeric(df_rates["startYear"], errors="coerce")  # остаётся float с NaN

    print("✅ df_rates очищен")

# ========== ФАЙЛ B: Страны и валюты (df_countries) ==========
if df_countries is not None:
    print("\n🧹 Очистка df_countries (страны и валюты)...")

    # 🔗 НЕ удаляем URL — переименовываем для удобства
    df_countries = df_countries.rename(columns={
        "country": "country_URL",   # ссылка на страну
        "currency": "currency_URL"  # ссылка на валюту
    }, errors="ignore")

    # Переименовываем Label-столбцы
    df_countries = df_countries.rename(columns={
        "countryLabel": "country",
        "currencyLabel": "currency"
    }, errors="ignore")

    # 📊 ИЗМЕРЯЕМ заполненность необязательных полей
    print("\n📈 Заполненность полей в df_countries:")
    if "population" in df_countries.columns:
        print(f"  • population: {df_countries['population'].notna().mean() * 100:.1f}% стран с населением")
    if "area" in df_countries.columns:
        print(f"  • area: {df_countries['area'].notna().mean() * 100:.1f}% стран с площадью")
    if "shortName" in df_countries.columns:
        print(f"  • shortName: {df_countries['shortName'].notna().mean() * 100:.1f}% валют с коротким кодом")

    # Числовые столбцы
    df_countries["population"] = pd.to_numeric(df_countries["population"], errors="coerce")  # float с NaN
    df_countries["area"] = pd.to_numeric(df_countries["area"], errors="coerce")  # float с NaN

    print("✅ df_countries очищен")

# ========== ИСХОДНЫЙ ФАЙЛ: currency.csv (если используется) ==========
if df_currency is not None:
    print("\n🧹 Очистка df_currency (исходный файл)...")

    # 🔗 Переименовываем, не удаляем
    df_currency = df_currency.rename(columns={"currency": "URL"}, errors="ignore")
    df_currency = df_currency.rename(columns={
        "currencyLabel": "currency",
        "unitLabel": "unit"
    }, errors="ignore")

    # Заполненность
    print("\n📈 Заполненность полей в df_currency:")
    if "year" in df_currency.columns:
        print(f"  • year: {df_currency['year'].notna().mean() * 100:.1f}% строк с годом")
    if "shortName" in df_currency.columns:
        print(f"  • shortName: {df_currency['shortName'].notna().mean() * 100:.1f}% с коротким кодом")

    # Числовые столбцы
    df_currency["price"] = pd.to_numeric(df_currency["price"], errors="coerce")
    df_currency["year"] = pd.to_numeric(df_currency["year"], errors="coerce")  # NaN остаётся

    print("✅ df_currency очищен")

print("\n✅ Очистка завершена. Столбцы с URL сохранены для отладки и объединения.")


🧹 Очистка df_rates (курсы валют)...

📈 Заполненность полей в df_rates:
  • year: 50.3% строк с годом
  • startYear: 1.1% строк с годом начала
  • shortName: 47.3% валют с коротким кодом
✅ df_rates очищен

🧹 Очистка df_countries (страны и валюты)...

📈 Заполненность полей в df_countries:
  • population: 99.8% стран с населением
  • area: 99.9% стран с площадью
  • shortName: 78.0% валют с коротким кодом
✅ df_countries очищен

✅ Очистка завершена. Столбцы с URL сохранены для отладки и объединения.


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame `df_currency`:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по числовым столбцам (`price`, `year`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы быстро получать сводку по данным.

**Столбцы в нашем датасете:**
- `currency` — название валюты (например, «иракский динар»)
- `shortName` — короткий код/аббревиатура
- `price` — цена валюты в евро (числовой)
- `year` — год данных (числовой)
- `unit` — единица измерения (например, «евро»)
- `unitSymbol` — символ единицы (например, «د.ع»)

In [5]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, столбцы, первые строки и статистика."""

    # 🔒 Проверка: DataFrame не должен быть None
    if df is None:
        print(f"⚠️ {name}: данные не загружены (None)")
        return

    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))

    numeric_cols = df.select_dtypes(include='number').columns
    if len(numeric_cols) > 0:
        print(f"\n📈 Статистика по числовым столбцам:")
        print(df[numeric_cols].describe())

# 🔍 Шаг 3. Обзор данных (с проверкой на None)
print("📋 Доступные данные для анализа:")

if df_rates is not None:
    show_info(df_rates, "Курсы валют (df_rates)")
else:
    print("⏭️ df_rates: файл не найден, пропускаем")

if df_countries is not None:
    show_info(df_countries, "Страны и валюты (df_countries)")
else:
    print("⏭️ df_countries: файл не найден, пропускаем")

# Если новые файлы не загружены — показываем исходный df_currency
if df_currency is not None:
    show_info(df_currency, "Валюты (df_currency) — исходный файл")
elif df_rates is None and df_countries is None:
    print("\n❌ Ошибка: ни один DataFrame не загружен!")

📋 Доступные данные для анализа:

📊 Курсы валют (df_rates)
Размер: (2346, 8)
Столбцы: URL, currency, shortName, price, year, startYear, unit, unitSymbol

Первые строки:
                                      URL         currency shortName  \
0  http://www.wikidata.org/entity/Q106720        вона КНДР       NaN   
1  http://www.wikidata.org/entity/Q122922   шведская крона       SEK   
2  http://www.wikidata.org/entity/Q123213  польский злотый       NaN   
3  http://www.wikidata.org/entity/Q123213  польский злотый       NaN   
4  http://www.wikidata.org/entity/Q123213  польский злотый       NaN   

          price    year  startYear            unit unitSymbol  
0  9.000000e-04  2018.0        NaN            евро          ₩  
1  9.718000e-02  2018.0        NaN            евро         kr  
2  1.800000e+06     NaN        NaN  польская марка         zł  
3  2.320000e-01  2018.0        NaN            евро         zł  
4  2.270350e-01  2020.0        NaN            евро         zł  

📈 Статистика п

## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** валют, единиц измерения и символов есть в данных;
- **какие единицы измерения встречаются чаще всего** (Топ‑5 по числу строк);
- **какие валюты представлены в датасете** (список уникальных названий);
- **диапазон цен и лет**: минимальная/максимальная цена валюты в евро, период данных.

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому
`df_currency["unit"].value_counts().head()` даёт **Топ‑5 единиц измерения по числу записей**.

> 💡 **Подсказка:** если в данных есть дубликаты (одна и та же валюта с разными символами), `value_counts()` поможет их быстро обнаружить.

In [6]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных")

# ========== ФАЙЛ A: Курсы валют ==========
if df_rates is not None:
    print("\n📊 Датасет: Курсы валют (df_rates)")
    print("Уникальных валют (по URL):", df_rates["URL"].nunique())
    print("Уникальных единиц измерения:", df_rates["unit"].nunique())

    # Цена: считаем только непустые значения
    valid_prices = df_rates["price"].dropna()
    if len(valid_prices) > 0:
        print(f"💰 Диапазон цен: {valid_prices.min():.6f} — {valid_prices.max():.6f} евро (на {len(valid_prices)} записях)")

    # Год: только непустые
    valid_years = df_rates["year"].dropna()
    if len(valid_years) > 0:
        print(f"📅 Диапазон лет: {int(valid_years.min())} — {int(valid_years.max())} (на {len(valid_years)} записях)")

    print("\n🏆 Топ-5 единиц измерения:")
    print(df_rates["unit"].value_counts().head())
else:
    print("\n⏭️ df_rates: пропускаем")

# ========== ФАЙЛ B: Страны и валюты ==========
if df_countries is not None:
    print("\n📊 Датасет: Страны и валюты (df_countries)")
    print("Уникальных стран (по URL):", df_countries["country_URL"].nunique())
    print("Уникальных валют (по URL):", df_countries["currency_URL"].nunique())

    # Население и площадь: только непустые
    valid_pop = df_countries["population"].dropna()
    valid_area = df_countries["area"].dropna()

    if len(valid_pop) > 0:
        print(f"👥 Население: {int(valid_pop.min()):,} — {int(valid_pop.max()):,} (на {len(valid_pop)} странах)")
    if len(valid_area) > 0:
        print(f"🗺️ Площадь: {valid_area.min():,.0f} — {valid_area.max():,.0f} км² (на {len(valid_area)} странах)")

    print("\n🏆 Топ-5 стран по числу записей:")
    print(df_countries["country"].value_counts().head())
else:
    print("\n⏭️ df_countries: пропускаем")

# ========== ИСХОДНЫЙ ФАЙЛ ==========
if df_currency is not None:
    print("\n📊 Датасет: Валюты (df_currency) — исходный файл")
    print("Уникальных валют (по URL):", df_currency["URL"].nunique())
    print("Уникальных единиц:", df_currency["unit"].nunique())

    valid_prices = df_currency["price"].dropna()
    if len(valid_prices) > 0:
        print(f"💰 Диапазон цен: {valid_prices.min():.6f} — {valid_prices.max():.6f} евро")

    print("\n🔣 Символы единиц:")
    print(df_currency["unitSymbol"].value_counts())
else:
    print("\n⏭️ df_currency: пропускаем")

print("\n✅ Валидация завершена")

🔍 Быстрая проверка данных

📊 Датасет: Курсы валют (df_rates)
Уникальных валют (по URL): 1012
Уникальных единиц измерения: 129
💰 Диапазон цен: 0.000001 — 1000000000.000000 евро (на 1569 записях)
📅 Диапазон лет: 1753 — 2026 (на 1179 записях)

🏆 Топ-5 единиц измерения:
unit
доллар США         361
евро               270
фунт стерлингов     70
датская крона       31
шведская крона      30
Name: count, dtype: int64

📊 Датасет: Страны и валюты (df_countries)
Уникальных стран (по URL): 209
Уникальных валют (по URL): 159
👥 Население: 882 — 1,477,519,529 (на 801 странах)
🗺️ Площадь: 0 — 17,075,400 км² (на 802 странах)

🏆 Топ-5 стран по числу записей:
country
Латвия                     38
Зимбабве                   32
Королевство Нидерландов    22
Франция                    20
Австрия                    19
Name: count, dtype: int64

⏭️ df_currency: пропускаем

✅ Валидация завершена


## 📋 Summary: что мы сделали в этом ноутбуке

✅ **Загрузили данные** из двух CSV-файлов:
- `currency_rates.csv` — курсы обмена валют с временными метками (Wikidata P2284, P585, P580)
- `countries_currencies.csv` — страны и их официальные валюты (Wikidata P38, P1082, P2046)

✅ **Очистили и переименовали столбцы**:
- Столбцы с URL Wikidata переименованы в `URL`, `country_URL`, `currency_URL` — сохранены для отладки и объединения таблиц
- Столбцы с постфиксом `Label` переименованы в читаемые названия: `currencyLabel → currency`, `unitLabel → unit`
- Числовые поля (`price`, `population`, `area`) приведены к типу `float`, текстовые — оставлены как есть

✅ **Оценили заполненность необязательных полей**:
- Вывели процент строк с данными для `year`, `startYear`, `shortName`, `population`, `area`
- Пустые значения (`NaN`) оставлены как есть — чтобы отличать «нет данных» от «значение равно нулю»

✅ **Проверили данные**:
- Посчитали уникальные значения по ключевым столбцам
- Посмотрели диапазоны числовых показателей только по непустым значениям
- Убедились, что столбцы с URL Wikidata сохранены для последующего объединения таблиц

🔜 **Далее (Задание 3)**: объединим таблицы по столбцу `currency_URL` / `URL`, построим графики динамики курсов и карту стран с цветовым кодированием по валюте.

> 💡 **Важно**: если в дальнейшем встретится аномальное значение (например, курс в миллиард евро), по столбцу `URL` можно в один клик перейти в Wikidata и проверить источник данных.